# SkinSense - Train the 12-class skin classifier on Colab

Trains **EfficientNet-B0** from the unified `dataset_manifest.csv` (DermNet + ACNE04 + HAM10000 + PAD-UFES-20) and
saves a checkpoint that drops straight into the backend via `WEIGHTS_PATH`.

**Before you start:** set Runtime -> Change runtime type -> **GPU (T4)**, then upload these 5 files to a Google Drive folder
(e.g. `MyDrive/SkinSense/`):

- `archive (1).zip`  (DermNet)
- `archive (2).zip`  (ACNE04)
- `archive (3).zip`  (HAM10000 + ISIC)
- `zr7vgbcyr2-1.zip` (PAD-UFES-20)
- `dataset_manifest.csv`

Then run the cells top to bottom.


## 1. GPU check + mount Drive


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU - set Runtime->GPU')
from google.colab import drive
drive.mount('/content/drive')


## 2. Paths + hyperparameters (edit `DRIVE_DIR` if needed)


In [ ]:
from pathlib import Path

DRIVE_DIR   = Path('/content/drive/MyDrive/SkinSense')   # folder with the 5 files
DERMNET_ZIP = DRIVE_DIR / 'archive (1).zip'
ACNE04_ZIP  = DRIVE_DIR / 'archive (2).zip'
HAM_ZIP     = DRIVE_DIR / 'archive (3).zip'
PADUFES_ZIP = DRIVE_DIR / 'zr7vgbcyr2-1.zip'
MANIFEST    = DRIVE_DIR / 'dataset_manifest.csv'

DATA_DIR    = Path('/content/data')          # extracted ImageFolder (local, fast)
OUT_CKPT    = DRIVE_DIR / 'best_model.pt'     # saved back to Drive

EPOCHS        = 30
WARMUP_EPOCHS = 3      # head-only, then unfreeze backbone
BATCH_SIZE    = 64
LR            = 3e-4
WEIGHT_DECAY  = 1e-4
PATIENCE      = 7      # early stop
MAX_PER_CLASS = 0      # 0 = use everything; set e.g. 1500 to cap / balance

CLASS_NAMES = ['acne','eczema','psoriasis','rosacea','seborrheic_keratoses','tinea',
               'melasma','vitiligo','hyperpigmentation','contact_dermatitis','warts','actinic_keratosis']
for p in [DERMNET_ZIP, ACNE04_ZIP, HAM_ZIP, PADUFES_ZIP, MANIFEST]:
    assert p.exists(), f'MISSING: {p}'
print('all inputs found')


## 3. Extract images from the manifest into an ImageFolder
Reads each labelled image straight from its zip (PAD-UFES nested zips are unpacked once) and writes
`/content/data/{split}/{label}/`. Runs once; a few minutes.


In [ ]:
import csv, io, os, zipfile, collections
from PIL import Image
import random

# Pre-unpack PAD-UFES nested image zips, index by filename.
pad_dir = Path('/content/padufes_imgs'); pad_dir.mkdir(exist_ok=True)
with zipfile.ZipFile(PADUFES_ZIP) as z:
    for iz in [n for n in z.namelist() if 'imgs_part' in n and n.endswith('.zip')]:
        with zipfile.ZipFile(io.BytesIO(z.read(iz))) as inner:
            inner.extractall(pad_dir)
pad_index = {}
for root, _, files in os.walk(pad_dir):
    for f in files:
        pad_index[f] = os.path.join(root, f)
print('PAD-UFES images unpacked:', len(pad_index))

handles = {
    'dermnet': zipfile.ZipFile(DERMNET_ZIP),
    'acne04':  zipfile.ZipFile(ACNE04_ZIP),
    'ham':     zipfile.ZipFile(HAM_ZIP),
}

rows = list(csv.DictReader(open(MANIFEST, encoding='utf-8')))
random.Random(42).shuffle(rows)
per_class = collections.Counter()
written = collections.Counter()
for i, r in enumerate(rows):
    src, ref, label, split = r['source'], r['ref'], r['label'], r['split']
    if MAX_PER_CLASS and split == 'train' and per_class[label] >= MAX_PER_CLASS:
        continue
    try:
        if src == 'padufes':
            path = pad_index.get(ref) or pad_index.get(Path(ref).name)
            if not path: continue
            img = Image.open(path).convert('RGB')
        else:
            img = Image.open(io.BytesIO(handles[src].read(ref))).convert('RGB')
    except Exception:
        continue
    d = DATA_DIR / split / label; d.mkdir(parents=True, exist_ok=True)
    img.save(d / f'{src}_{i}.jpg', 'JPEG', quality=92)
    if split == 'train': per_class[label] += 1
    written[split] += 1
print('written:', dict(written))
for c in sorted(os.listdir(DATA_DIR/'train')):
    tr = len(os.listdir(DATA_DIR/'train'/c)); va = len(os.listdir(DATA_DIR/'val'/c)) if (DATA_DIR/'val'/c).exists() else 0
    print(f'  {c:22s} train={tr:5d} val={va:4d}')


## 4. Build model + data loaders (fixes: absent-class weight=0, portable 12-output head)


In [ ]:
import torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
train_tf = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20), transforms.ColorJitter(0.2,0.2,0.2,0.02),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
val_tf = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

train_ds = datasets.ImageFolder(DATA_DIR/'train', transform=train_tf)
val_ds   = datasets.ImageFolder(DATA_DIR/'val',   transform=val_tf)
# Remap ImageFolder's local labels to the GLOBAL 12-class index so the checkpoint is portable.
for ds in (train_ds, val_ds):
    remap = {li: CLASS_NAMES.index(n) for n, li in ds.class_to_idx.items()}
    ds.target_transform = (lambda y, _m=remap: _m[y])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

class TemperatureScaler(nn.Module):
    def __init__(self, base, t=1.0):
        super().__init__(); self.base_model = base
        self.temperature = nn.Parameter(torch.tensor(float(t)), requires_grad=False)
    def forward(self, x): return self.base_model(x) / self.temperature

backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
backbone.classifier[1] = nn.Linear(backbone.classifier[1].in_features, len(CLASS_NAMES))
model = TemperatureScaler(backbone, 1.0).to(device)

# Class weights over the 12-slot space; ABSENT classes get weight 0 (critical).
counts = torch.zeros(len(CLASS_NAMES))
l2g = {li: CLASS_NAMES.index(n) for n, li in train_ds.class_to_idx.items()}
for _, ll in train_ds.samples: counts[l2g[ll]] += 1
present = counts > 0
w = torch.zeros(len(CLASS_NAMES)); w[present] = counts[present].sum() / (present.sum() * counts[present])
criterion = nn.CrossEntropyLoss(weight=w.to(device), label_smoothing=0.05)
print('train', len(train_ds), 'val', len(val_ds), 'present classes:', int(present.sum().item()))


## 5. Train (warmup head, then fine-tune the whole backbone)


In [ ]:
def set_backbone_trainable(m, flag):
    for n, p in m.base_model.named_parameters():
        p.requires_grad = True if n.startswith('classifier') else flag

@torch.no_grad()
def evaluate():
    model.eval(); nc = len(CLASS_NAMES)
    cc = torch.zeros(nc); ct = torch.zeros(nc); correct = total = 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device); pred = model(x).argmax(1)
        correct += (pred==y).sum().item(); total += y.numel()
        for c in range(nc):
            msk = y==c; ct[c]+=msk.sum().item(); cc[c]+=(pred[msk]==c).sum().item()
    rec = cc/ct.clamp(min=1)
    return correct/max(total,1), rec[ct>0].mean().item()

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=device.type=='cuda')
set_backbone_trainable(model, False)
best = -1.0; no_improve = 0
for epoch in range(1, EPOCHS+1):
    if epoch == WARMUP_EPOCHS+1:
        set_backbone_trainable(model, True)
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-epoch+1))
        print('-- backbone unfrozen --')
    model.train(); run = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device); opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=device.type=='cuda'):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); run += loss.item()*y.size(0)
    sched.step()
    acc, macro = evaluate()
    print(f'epoch {epoch:2d}/{EPOCHS}  loss={run/len(train_ds):.4f}  val_acc={acc:.4f}  val_macro_recall={macro:.4f}')
    if macro > best:
        best = macro; no_improve = 0; torch.save(model.state_dict(), OUT_CKPT)
        print(f'   saved best -> {OUT_CKPT} (macro={macro:.4f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE: print('early stop'); break
print('best macro_recall =', best)


## 6. Calibrate temperature (honest confidences) + save


In [ ]:
model.load_state_dict(torch.load(OUT_CKPT, map_location=device)); model.eval()
logits, labels = [], []
with torch.no_grad():
    for x, y in val_loader:
        logits.append(model.base_model(x.to(device)).cpu()); labels.append(y)
logits = torch.cat(logits); labels = torch.cat(labels)
T = torch.nn.Parameter(torch.ones(1))
optT = torch.optim.LBFGS([T], lr=0.01, max_iter=100)
def closure():
    optT.zero_grad(); l = F.cross_entropy(logits/T.clamp(min=0.05), labels); l.backward(); return l
optT.step(closure)
raw = float(T.detach().item()); t = min(max(raw,0.5),5.0)
if t in (0.5,5.0): print(f'calibration hit bound (raw={raw:.3f}); using 1.0'); t = 1.0
with torch.no_grad(): model.temperature.copy_(torch.tensor(float(t)))
torch.save(model.state_dict(), OUT_CKPT)
print(f'temperature={t:.3f}  saved -> {OUT_CKPT}')


## 7. Serve it
Download `best_model.pt` from your Drive folder and point the backend at it:
```bash
WEIGHTS_PATH=/abs/path/best_model.pt
```
The checkpoint is a `TemperatureScaler` state_dict with a 12-output head in `CLASS_NAMES` order, so
`load_model()` loads it with `strict=True`. Grad-CAM will now produce real activations.

**Note:** rosacea, melasma, vitiligo, hyperpigmentation, and contact_dermatitis have no rows in the manifest
(these datasets can't label them cleanly). The model outputs all 12 classes but will never predict those 5
until you add images for them (Fitzpatrick17k or your own) and rebuild the manifest.
